# TorchSig Training Utilities Demo

**Created:** 2026-06-24

**Purpose:** Demonstrates a full TorchSig → EfficientNet training pipeline with TensorBoard and MLflow integration.

---

### Imports
All required libraries, grouped by category (standard, third-party, PyTorch, TorchSig, and custom models).

In [ ]:
import sys
import os

# Add the path to the ROOT of the torchsig-github folder
sys.path.insert(0, "../torchsig-github")

In [ ]:
import torchsig
print(torchsig.__version__)

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
import mlflow
import torch

from torchsig.datasets.datasets import TorchSigIterableDataset, StaticTorchSigDataset
from torchsig.transforms.transforms import ComplexTo2D, Spectrogram
from torchsig.utils.data_loading import WorkerSeedingDataLoader
from torchsig.utils.defaults import TorchSigDefaults
from torchsig.utils.writer import DatasetCreator

from torchsig_models.models.iq_models.efficientnet import efficientnet_b4
from torchsig_models.util.training import (
    compute_class_weights_tensor,
    compute_num_params,
    evaluate_classifier,
    set_deterministic,
    train_validate,
)

### Configuration
Define experiment settings, training hyperparameters, dataset parameters, and output locations in a single place. This makes experiments reproducible and simplifies parameter tuning.

In [ ]:
config = {
    "dataset_splits": [0.7, 0.2, 0.1],
    "overwrite": True,
    "num_workers": 2,
    "dataset_length": 10_000,
    "seed": 123,
    "dataset_id": "iq_narrowband",
    "impairment_level": 0,
    "output_representation": "iq",
}

params = {
    "experiment_name": "torchsig-efficientnet-IQ-1d-narrowband",
    "model_name": "efficientnet_b4",
    "batch_size": 32,
    "max_epochs": 30,
    "learning_rate": 1e-3,
    "weight_decay": 5e-3,
    "label_smoothing": 0.1,
    "drop_path_rate": 0.2,
    "drop_rate": 0.3,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Reproducibility
Configure Python, NumPy, and PyTorch random number generators to improve experiment reproducibility. This helps ensure that dataset generation, training, and evaluation produce consistent results across runs.

In [ ]:
set_deterministic(config["seed"])

### Environment Setup
Initialize the training environment by loading configuration, setting deterministic seeds, creating output directories, and reporting hardware availability.

In [ ]:
torch.cuda.empty_cache()
load_dotenv(Path(".env"))
set_deterministic(config["seed"])

root = Path("datasets") / config["dataset_id"]
checkpoint_dir = Path("run_configs") / config["dataset_id"] / "checkpoints"
metrics_dir = Path("experiment_metrics") / config["dataset_id"]

for path in [root, checkpoint_dir, metrics_dir]:
    path.mkdir(parents=True, exist_ok=True)

config["root"] = str(root)

print("TRACKING_URI :", os.getenv("MLFLOW_TRACKING_URI"))
print("CA bundle    :", os.getenv("REQUESTS_CA_BUNDLE"))
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")

### Dataset Metadata
Configure TorchSIG dataset generation parameters, including signal characteristics, frequency ranges, bandwidth limits, and sample dimensions used during synthetic data generation.

In [ ]:
fft_size = 256

dataset_metadata = TorchSigDefaults().default_dataset_metadata.copy()
dataset_metadata.update(
    {
        "num_iq_samples_dataset": fft_size**2,
        "fft_size": fft_size,
        "fft_stride": fft_size,
        "num_signals_max": 1,
        "num_signals_min": 1,
        "noise_power_db": 1,
        "signal_center_freq_min": 1000,
        "signal_center_freq_max": 2000,
        "sample_rate": 10000,
        "frequency_min": 1000,
        "frequency_max": 2000,
        "cochannel_overlap_probability": 0.2,
        "bandwidth_min": 1000,
        "bandwidth_max": 2000,
    }
)

### Transforms
Select the model input representation. Signals can be provided directly as IQ samples or converted into alternative representations such as spectrograms.

In [ ]:
if config["output_representation"].lower() == "iq":
    transforms = [ComplexTo2D()]
else:
    raise ValueError(
        f"Unsupported output representation: {config['output_representation']}"
    )

### Dataset Generation
Generate train, validation, and test datasets from TorchSIG signal generators and save them to disk for reproducible experimentation.

In [ ]:
train_len = int(config["dataset_length"] * config["dataset_splits"][0])
val_len = int(config["dataset_length"] * config["dataset_splits"][1])
test_len = config["dataset_length"] - train_len - val_len

split_lengths = {
    "train": train_len,
    "val": val_len,
    "test": test_len,
}

iterable_datasets = {
    split: TorchSigIterableDataset(
        metadata=dataset_metadata,
        transforms=transforms,
        target_labels=None,
        signal_generators="all",
    )
    for split in split_lengths
}

config["classes"] = iterable_datasets["train"].class_names
config["num_classes"] = len(config["classes"])

for split, dataset in iterable_datasets.items():
    dataloader = WorkerSeedingDataLoader(
        dataset,
        batch_size=params["batch_size"],
        collate_fn=lambda x: x,
        num_workers=config["num_workers"],
    )

    DatasetCreator(
        dataloader=dataloader,
        root=str(root / split),
        overwrite=config["overwrite"],
        dataset_length=split_lengths[split],
    ).create()

### Dataset Loading
Load the generated datasets and create PyTorch dataloaders for efficient training, validation, and testing.

In [ ]:
static_datasets = {
    split: StaticTorchSigDataset(
        root=str(root / split),
        target_labels=["class_index"],
    )
    for split in split_lengths
}

train_loader = WorkerSeedingDataLoader(
    static_datasets["train"],
    batch_size=params["batch_size"],
    shuffle=True,
    num_workers=config["num_workers"],
)
val_loader = WorkerSeedingDataLoader(
    static_datasets["val"],
    batch_size=params["batch_size"],
    num_workers=config["num_workers"],
)
test_loader = WorkerSeedingDataLoader(
    static_datasets["test"],
    batch_size=params["batch_size"],
    num_workers=config["num_workers"],
)

In [ ]:
train_dataset = static_datasets["train"]

### Model, loss, optimizer, scheduler
Construct the neural network, define the training objective, configure optimization, and specify the learning-rate schedule used during training.

In [ ]:
class_weights = compute_class_weights_tensor(
    train_loader,
    config["num_classes"],
).to(DEVICE)

model = efficientnet_b4(
    num_classes=config["num_classes"],
    drop_path_rate=params["drop_path_rate"],
    drop_rate=params["drop_rate"],
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=params["label_smoothing"],
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=params["learning_rate"],
    weight_decay=params["weight_decay"],
)

warmup_epochs = min(5, params["max_epochs"])
scheduler_warm = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    total_iters=warmup_epochs,
)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(params["max_epochs"] - warmup_epochs, 1),
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[scheduler_warm, scheduler_cos],
    milestones=[warmup_epochs],
)

params["num_params"] = compute_num_params(model)
print(f"Trainable parameters: {params['num_params']:,}")

### Training
Train the model using PyTorch Lightning while tracking loss, accuracy, precision, recall, F1 score, and confusion matrices through the metrics callback.

In [ ]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment(params["experiment_name"])

with mlflow.start_run():
    mlflow.log_params({**config, **params})
    mlflow.log_param("classes", config["classes"])

    pl_model, metrics = train_validate(
        train_loader=train_loader,
        val_loader=val_loader,
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        max_epochs=params["max_epochs"],
        num_classes=config["num_classes"],
        metrics_dir=metrics_dir,
        checkpoint_dir=checkpoint_dir,
        accelerator="auto",
        devices="auto",
        precision="32-true",
        use_distributed_sampler=False,
        enable_progress_bar=True,
    )

    metrics.save_to_csv()
    metrics.plot(save_dir=metrics_dir, show=True, close=False)

### Load best checkpoint
Load the highest-performing checkpoint based on validation F1 score to ensure evaluation is performed using the best available model.

In [ ]:
best_ckpts = list(checkpoint_dir.glob("best-epoch*.ckpt"))
if not best_ckpts:
    raise FileNotFoundError(f"No best checkpoint found in {checkpoint_dir}")

best_ckpt_path = max(best_ckpts, key=lambda path: path.stat().st_mtime)
checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)

model_state_dict = {
    key.removeprefix("model."): value
    for key, value in checkpoint["state_dict"].items()
    if key.startswith("model.")
}

model.load_state_dict(model_state_dict, strict=True)

### Testing
Evaluate the trained model on the held-out test set and compute final performance metrics and confusion matrices.

In [ ]:
test_tracker = evaluate_classifier(
    model=model,
    test_loader=test_loader,
    device=DEVICE,
    num_classes=config["num_classes"],
    criterion=criterion,
)

test_tracker.save_to_csv(metrics_dir / "test")

test_tracker.plot(
    save_dir=metrics_dir / "test",
    prefix="test ",
    show=True,
    close=False,
)

In [ ]:
test_tracker.plot_confusion_matrix(
    save_file=metrics_dir / "test" / "confusion_matrix.png",
    class_names=config["classes"],
)

test_tracker.plot_confusion_matrix(
    normalize=True,
    save_file=metrics_dir / "test" / "confusion_matrix_normalized.png",
    class_names=config["classes"],
)

### MLflow Logging

Record experiment parameters, training metrics, and test results to MLflow for experiment tracking and reproducibility.

In [ ]:
mlflow.log_metrics(
    {
        "test_loss": test_tracker.history["loss"][-1],
        "test_accuracy": test_tracker.history["accuracy"][-1],
        "test_f1": test_tracker.history["f1 score"][-1],
        "test_precision": test_tracker.history["precision"][-1],
        "test_recall": test_tracker.history["recall"][-1],
    }
)
mlflow.end_run()